In [195]:
!jupyter nbconvert --to notebook --execute preprocess.ipynb --inplace

[NbConvertApp] Converting notebook preprocess.ipynb to notebook
[NbConvertApp] Writing 116043 bytes to preprocess.ipynb


In [385]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import poisson
from pulp import *

In [386]:
OUT_CSV = "fantasy_enriched.csv"



| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |



## Core identity / player info

| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | Unique FIFA fantasy player ID |
| `name` | str | Player name |
| `squad_id` | int | Internal squad/nation ID |
| `team` | str | National team |
| `position` | str | DEF / MID / FWD / GK |
| `price` | float | Fantasy price |
| `status` | str | Availability status |
| `total_points` | int | Total tournament points |
| `avg_points` | float | Average points per match |
| `matches_played` | int | Matches with participation |
| `percent_selected` | float | Selection rate (%) |
| `next_fixture` | int | Next match ID |


## Round fantasy points

| Column | Type | Description |
|---|---|---|
| `round_1_points` | int | Points in round 1 |
| `round_2_points` | int | Points in round 2 |
| `round_3_points` | int | Points in round 3 |
| `round_4_points` | int | Points in round 4 |


## Betting / scorer mapping

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Matched Betano market name |
| `betano_match_score` | float | Name matching confidence (0–100) |
| `anytime_scorer_prob` | float | Probability player scores anytime |
| `first_scorer_prob` | float | Probability scores first goal |
| `last_scorer_prob` | float | Probability scores last goal |


## Match context

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team |
| `match_away` | str | Away team |
| `is_home` | bool | Player team is home |
| `opponent` | str | Opponent team |


## Match probabilities 

| Column | Type | Description |
|---|---|---|
| `match_home_win_prob` | float | Home win probability |
| `match_draw_prob` | float | Draw probability |
| `match_away_win_prob` | float | Away win probability |
| `match_btts_prob` | float | Both teams score probability |
| `match_over_25_prob` | float | Over 2.5 goals probability |
| `match_over_35_prob` | float | Over 3.5 goals probability |
| `team_score_prob` | float | Team scores ≥1 goal probability |
| `team_score_2_prob` | float | Team scores ≥2 goals probability |
| `team_cs_prob` | float | Team clean sheet probability |
| `opp_score_prob` | float | Opponent scores ≥1 goal probability |
| `opp_cs_prob` | float | Opponent clean sheet probability |
| `team_over_05_prob` | float | Team scores ≥1 goal probability |
| `team_over_15_prob` | float | Team scores ≥2 goals probability |
| `team_qualify_prob` | float | Team advances probability |
| `team_win2_prob` | float | Team wins by 2+ goals probability |
| `opp_over_05_prob` | float | Opponent scores ≥1 goal probability |


## Raw cumulative stats

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals |
| `stat_AS` | int | Assists |
| `stat_CS` | int | Clean sheets |
| `stat_GC` | int | Goals conceded |
| `stat_MP` | int | Minutes played |
| `stat_YC` | int | Yellow cards |
| `stat_RC` | int | Red cards |
| `stat_ST` | int | Shots on target |
| `stat_SB` | int | Shots blocked |
| `stat_CC` | int | Chances created |
| `stat_PS` | int | Penalties saved |
| `stat_T` | int | Tackles |
| `stat_S` | int | Saves |
| `stat_SXI` | int | Starts (XI appearances) |
| `stat_OG` | int | Own goals |
| `stat_PC` | int | Penalties committed |
| `stat_PW` | int | Penalties won |
| `stat_FK` | int | Free kicks won |



## Round-by-round stats (1–4)

| Pattern | Type | Description |
|---|---|---|
| `round_i_SXI` | int | Started (0/1) |
| `round_i_MP` | int | Minutes |
| `round_i_AS` | int | Assists |
| `round_i_YC` | int | Yellow cards |
| `round_i_RC` | int | Red cards |
| `round_i_OG` | int | Own goals |
| `round_i_PW` | int | Penalties won |
| `round_i_PC` | int | Penalties committed |
| `round_i_CS` | int | Clean sheets |
| `round_i_GS` | int | Goals |
| `round_i_GC` | int | Goals conceded |
| `round_i_PS` | int | Penalties saved |
| `round_i_T` | int | Tackles |
| `round_i_CC` | int | Chances created |
| `round_i_ST` | int | Shots on target |
| `round_i_FK` | int | Free kicks |
| `round_i_S` | int | Saves |
| `round_i_SB` | int | Shots blocked |



## External matching / enrichment

| Column | Type | Description |
|---|---|---|
| `clubstats_matched_player` | str | External player match |
| `clubs_match_dist` | float | Matching distance score |



## Form / usage trends

| Column | Type | Description |
|---|---|---|
| `started_last_match` | int | Started last match |
| `starts_last_3` | int | Starts last 3 matches |
| `starts_last_5` | int | Starts last 5 matches |
| `start_rate` | float | Start frequency |
| `minutes_last_3_avg` | float | Avg minutes last 3 |
| `minutes_last_5_avg` | float | Avg minutes last 5 |



## Performance rates

| Column | Type | Description |
|---|---|---|
| `goal_rate` | float | Goals per minute |
| `goals_per_start` | float | Goals per start |
| `shots_on_target_per90` | float | SOT per 90 |
| `goals_per_shot_on_target` | float | Conversion rate |
| `assist_rate` | float | Assists per minute |
| `chance_created_per90` | float | Chances per 90 |
| `tackles_per90` | float | Tackles per 90 |
| `cc_per90` | float | Chances created per 90 |
| `sot_per90` | float | Shots on target per 90 |
| `saves_per90` | float | Saves per 90 |
| `yc_per90` | float | Yellow cards per 90 |
| `rc_per90` | float | Red cards per 90 |
| `penalty_conceded_rate` | float | Penalties conceded per 90 |
| `own_goal_rate` | float | Own goals per 90 |
| `penalties_won_per90` | float | Penalties won per 90 |



## Recent / streak features

| Column | Type | Description |
|---|---|---|
| `recent_goals_last3` | int | Goals last 3 matches |
| `goal_streak` | int | Consecutive scoring matches |
| `recent_assists_last3` | int | Assists last 3 matches |
| `recent_chances_created_per90` | float | Recent creation rate |
| `recent_cs_rate` | float | Clean sheet rate |
| `recent_tackles_per90` | float | Recent tackles |
| `recent_cc_per90` | float | Recent chances created |
| `recent_sot_per90` | float | Recent shots on target |
| `recent_saves_per90` | float | Recent saves |
| `recent_penalties_won` | int | Penalties won last 3 |



## Expected value features

| Column | Type | Description |
|---|---|---|
| `goal_expectation` | float | Scoring proxy |
| `goal_expectation2` | float | Scoring proxy (2+ goals) |
| `assist_expectation` | float | Assist proxy |
| `cs_expectation` | float | Clean sheet proxy |
| `expected_minutes` | float | Expected minutes |
| `expected_gc_penalty` | float | Defensive penalty |
| `expected_tackle_points` | float | Tackles contribution |
| `expected_cc_points` | float | Chance creation points |
| `expected_sot_points` | float | Shot contribution points |
| `expected_save_points` | float | Save contribution points |



## Fantasy points dynamics

| Column | Type | Description |
|---|---|---|
| `points_last1` | int | Last match points |
| `points_last3_avg` | float | Avg last 3 |
| `points_last5_avg` | float | Avg last 5 |
| `weighted_points` | float | Recency weighted |
| `rolling_std_points` | float | Variance last 5 |
| `rolling_max_points` | int | Max last 5 |



## Team-position ranking features

| Column | Type | Description |
|---|---|---|
| `price_rank_team_position` | float | Price rank |
| `selected_rank_team_position` | float | Selection rank |
| `points_rank_team_position` | float | Points rank |
| `minutes_rank_team_position` | float | Minutes rank |
| `starts_rank_team_position` | float | Starts rank |
| `anytime_rank_team_position` | float | Scoring odds rank |
| `shots_rank_team_position` | float | Shooting rank |
| `cc_rank_team_position` | float | Chance creation rank |
| `tackles_rank_team_position` | float | Tackles rank |

In [387]:
df= pd.read_csv("fantasy_enriched.csv")

In [388]:
df = df[df["status"] == "playing"].copy()

In [389]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [390]:
# probability of playing a minute

raw_any = (
    0.15 * df["minutes_rank_team_position"]
  + 0.20 * df["stat_MP"]        # minutes needs to be very team stratified since while for other point sources players are all eligible
  + 0.10 * df["starts_rank_team_position"]
  + 0.15 * df["price_rank_team_position"]     # for minutes its a team stratified signal.
  + 0.20 * df["selected_rank_team_position"]
  + 0.15 * df["minutes_last_3_avg"]
  + 0.05 * df["anytime_rank_team_position"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_any"] = sigmoid((raw_any - 0.5) * 6)

In [391]:
df.loc[
    :,
    [
        "prob_plays_any",
        "name",
        "team",
        "position",
        "stat_MP",
        "minutes_rank_team_position",
        "starts_rank_team_position",
        "selected_rank_team_position",
        "price_rank_team_position",
        "anytime_rank_team_position",
    ],
].sort_values("prob_plays_any", ascending=False).head(30)

,prob_plays_any,name,team,position,stat_MP,minutes_rank_team_position,starts_rank_team_position,selected_rank_team_position,price_rank_team_position,anytime_rank_team_position
340,0.947350,Yassine Bounou,Morocco,GK,1.000000,1.000000,1.000000,1.00,1.000000,0.633333
272,0.947350,Thibaut Courtois,Belgium,GK,1.000000,1.000000,1.000000,1.00,1.000000,0.633333
106,0.940643,Kylian Mbappé,France,FWD,0.900000,1.000000,1.000000,1.00,1.000000,1.000000
278,0.937479,Diogo Costa,Portugal,GK,0.923077,1.000000,1.000000,1.00,1.000000,0.633333
110,0.937479,Mike Maignan,France,GK,0.923077,1.000000,1.000000,1.00,1.000000,0.633333
49,0.937479,Maxime Crépeau,Canada,GK,0.923077,1.000000,1.000000,1.00,1.000000,0.633333
91,0.937479,Jordan Pickford,England,GK,0.923077,1.000000,1.000000,1.00,1.000000,0.633333
85,0.929234,Harry Kane,England,FWD,0.907692,1.000000,0.656250,1.00,1.000000,1.000000
224,0.927081,Unai Simón,Spain,GK,0.923077,1.000000,1.000000,1.00,0.816667,0.633333
170,0.926121,Gustavo Gómez,Paraguay,DEF,1.000000,1.000000,0.214286,1.00,1.000000,1.000000


In [392]:
# probability of starting / playing 60 minutes

raw_any = (
    0.15 * df["minutes_rank_team_position"]
  + 0.10 * df["starts_rank_team_position"]
  + 0.25 * df["stat_MP"]
  + 0.10 * df["price_rank_team_position"]
  + 0.05 * df["selected_rank_team_position"]
  + 0.15 * df["minutes_last_3_avg"]
  + 0.15 * df["round_4_MP"]
  + 0.05 * df["anytime_rank_team_position"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_60min"] = sigmoid((raw_any - 0.5) * 6)


In [393]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "minutes_rank_team_position",
    "starts_rank_team_position",
    "price_rank_team_position",
    "selected_rank_team_position",
    "minutes_last_3_avg",
    "round_4_MP",
    "anytime_rank_team_position",
]

df[cols].sort_values("prob_plays_60min", ascending=False).head(30)

,name,team,position,prob_plays_60min,minutes_rank_team_position,starts_rank_team_position,price_rank_team_position,selected_rank_team_position,minutes_last_3_avg,round_4_MP,anytime_rank_team_position
272,Thibaut Courtois,Belgium,GK,0.947350,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.633333
340,Yassine Bounou,Morocco,GK,0.947350,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.633333
170,Gustavo Gómez,Paraguay,DEF,0.926121,1.000000,0.214286,1.000000,1.000000,1.000000,1.000000,1.000000
341,Achraf Hakimi,Morocco,DEF,0.923039,1.000000,0.140625,1.000000,1.000000,1.000000,1.000000,1.000000
106,Kylian Mbappé,France,FWD,0.922048,1.000000,1.000000,1.000000,1.000000,0.870000,0.708333,1.000000
49,Maxime Crépeau,Canada,GK,0.921262,1.000000,1.000000,1.000000,1.000000,0.900000,0.750000,0.633333
91,Jordan Pickford,England,GK,0.921262,1.000000,1.000000,1.000000,1.000000,0.900000,0.750000,0.633333
110,Mike Maignan,France,GK,0.921262,1.000000,1.000000,1.000000,1.000000,0.900000,0.750000,0.633333
278,Diogo Costa,Portugal,GK,0.921262,1.000000,1.000000,1.000000,1.000000,0.900000,0.750000,0.633333
186,Orlando Gill,Paraguay,GK,0.920561,1.000000,1.000000,0.266667,1.000000,1.000000,1.000000,0.633333


There is a phenomenon where a player will be overvalued, such as Lukaku here for belgium. He has not played much, but since Belgium only has 3 forward and only him has significative minutes he ranks number 1 for all ranks in BEL,FWD. Need to include more general sums and beware with manual selection.

In [394]:
raw_goal = (
    0.60 * df["anytime_scorer_prob"]
  + 0.05 * df["recent_goals_last3"]
  + 0.05 * df["team_score_2_prob"]
  + 0.02 * df["stat_GS"]
  + 0.03 * df["stat_ST"]
  + 0.05 * df["price"]
)

df["prob_scores"] = raw_goal * df["prob_plays_60min"]

In [395]:
cols = [
    "name",
    "team",
    "position",
    "prob_scores",
    "goal_expectation",
    "anytime_scorer_prob",
    "recent_goals_last3",
    "recent_sot_per90",
    "goal_rate",
    "shots_on_target_per90",
    "stat_GS",
    "stat_ST",
    "price_rank_team_position",
]

df[cols].sort_values("prob_scores", ascending=False).head(30)

,name,team,position,prob_scores,goal_expectation,anytime_scorer_prob,recent_goals_last3,recent_sot_per90,goal_rate,shots_on_target_per90,stat_GS,stat_ST,price_rank_team_position
106,Kylian Mbappé,France,FWD,0.737638,1.000000,1.000000,1.00,0.402299,0.307692,0.518519,1.000000,1.000000,1.000000
220,Mikel Oyarzabal,Spain,FWD,0.515971,0.663987,0.736435,1.00,0.387046,0.239203,0.372093,0.666667,0.615385,1.000000
27,Vinícius Júnior,Brazil,MID,0.512489,0.664927,0.718813,0.75,0.402299,0.205128,0.398860,0.666667,0.769231,1.000000
157,Erling Haaland,Norway,FWD,0.491637,0.548897,0.725769,0.75,0.324074,0.333333,0.466667,0.833333,0.692308,1.000000
85,Harry Kane,England,FWD,0.478226,0.506153,0.633792,0.75,0.265152,0.254237,0.355932,0.833333,0.692308,1.000000
107,Ousmane Dembélé,France,MID,0.466114,0.710163,0.708610,1.00,0.280449,0.250000,0.243056,0.666667,0.384615,1.000000
282,Cristiano Ronaldo,Portugal,FWD,0.424381,0.435538,0.584171,0.75,0.312899,0.153846,0.279202,0.500000,0.538462,1.000000
10,Romelu Lukaku,Belgium,FWD,0.419346,0.623027,0.718813,0.50,0.152505,0.203390,0.158192,0.333333,0.153846,1.000000
28,Matheus Cunha,Brazil,FWD,0.398705,0.586817,0.633792,0.75,0.283172,0.229787,0.297872,0.500000,0.384615,1.000000
144,Ismael Saibari,Morocco,MID,0.386342,0.492985,0.562220,0.50,0.085158,0.148760,0.115702,0.500000,0.230769,1.000000


In [396]:
prob_assists = (
    0.05 * df["chance_created_per90"]
  + 0.20 * df["stat_CC"]
  + 0.05 * df["recent_cc_per90"]
  + 0.15 * df["team_score_2_prob"]
  + 0.15 * df["recent_assists_last3"]
  + 0.15 * df["price"]
  + 0.10 * df["stat_AS"]
)

df["expected_assists"] = prob_assists * df["prob_plays_60min"]

In [397]:
cols = [
    "name",
    "team",
    "position",
    "expected_assists",
    "assist_expectation",
    "stat_CC",
    "recent_cc_per90",
    "team_score_2_prob",
    "recent_assists_last3",
    "price",
    "prob_plays_60min",
    "stat_AS",
]

df[cols].sort_values("expected_assists", ascending=False).head(30)

,name,team,position,expected_assists,assist_expectation,stat_CC,recent_cc_per90,team_score_2_prob,recent_assists_last3,price,prob_plays_60min,stat_AS
119,Michael Olise,France,MID,0.599446,0.172615,1.0,0.178899,1.000000,1.00,0.857143,0.803942,1.0
106,Kylian Mbappé,France,FWD,0.423141,0.030294,0.2,0.049808,1.000000,0.50,1.000000,0.922048,0.4
252,Bruno Guimarães,Brazil,MID,0.398990,0.086657,0.6,0.150000,0.704250,0.75,0.471429,0.796764,0.8
216,Marc Cucurella,Spain,DEF,0.360932,0.079799,0.6,0.096296,0.643539,0.75,0.228571,0.834914,0.6
107,Ousmane Dembélé,France,MID,0.350990,0.036920,0.2,0.062500,1.000000,0.50,0.928571,0.781460,0.4
27,Vinícius Júnior,Brazil,MID,0.302459,0.027980,0.2,0.049808,0.704250,0.25,0.928571,0.873305,0.2
130,Roberto Alvarado,Mexico,FWD,0.293765,0.091757,0.8,0.156000,0.289853,0.50,0.257143,0.751832,0.6
98,Jude Bellingham,England,MID,0.290945,0.081042,0.6,0.166667,0.398092,0.25,0.685714,0.823783,0.2
293,Martin Ødegaard,Norway,MID,0.270929,0.061634,0.4,0.072222,0.338248,0.50,0.600000,0.745991,0.6
341,Achraf Hakimi,Morocco,DEF,0.260691,0.047676,0.4,0.086667,0.563053,0.25,0.357143,0.923039,0.2


In [398]:
position_yc_modifier = {
    "GK": 0.15,
    "FWD": 0.35,
    "DEF": 0.45,
    "MID": 0.55,
}

df["position_yc_modifier"] = df["position"].map(position_yc_modifier)

raw_yc = (
    0.35 * df["yc_per90"]
  + 0.15 * df["tackles_per90"]
  + 0.15 * df["opp_over_05_prob"]
  + 0.05 * df["rc_per90"]
  + 0.05 * df["position_yc_modifier"]
)

df["prob_yellow_card"] = sigmoid((raw_yc - 0.6) * 5) * df["prob_plays_60min"]

In [399]:
cols = [
    "name",
    "team",
    "position",
    "prob_yellow_card",
    "yc_per90",
    "stat_YC",
    "tackles_per90",
    "recent_tackles_per90",
    "opp_over_05_prob",
    "prob_plays_60min",
    "rc_per90",
]

df[cols].sort_values("prob_yellow_card", ascending=False).head(30)

,name,team,position,prob_yellow_card,yc_per90,stat_YC,tackles_per90,recent_tackles_per90,opp_over_05_prob,prob_plays_60min,rc_per90
168,Matías Galarza,Paraguay,MID,0.130667,0.255034,1.0,0.046980,0.281879,1.000000,0.798212,0.000000
284,Andrés Cubas,Paraguay,MID,0.104211,0.097436,0.5,0.035897,0.180000,1.000000,0.811533,0.000000
177,Miguel Almirón,Paraguay,MID,0.099216,0.177570,0.5,0.032710,0.222222,1.000000,0.588171,0.728972
171,Júnior Alonso,Paraguay,DEF,0.098627,0.126667,0.5,0.026667,0.171429,1.000000,0.755296,0.000000
170,Gustavo Gómez,Paraguay,DEF,0.098385,0.000000,0.0,0.010256,0.060000,1.000000,0.926121,0.000000
175,Juan José Cáceres,Paraguay,DEF,0.097152,0.108883,0.5,0.051576,0.288889,1.000000,0.752093,0.000000
10,Romelu Lukaku,Belgium,FWD,0.095095,0.214689,0.5,0.000000,0.000000,0.706667,0.802544,0.000000
186,Orlando Gill,Paraguay,GK,0.090798,0.000000,0.0,0.000000,0.000000,1.000000,0.920561,0.000000
303,Alistair Johnston,Canada,DEF,0.089558,0.105556,0.5,0.002778,0.000000,0.760766,0.843267,0.000000
7,Brandon Mechele,Belgium,DEF,0.087385,0.097436,0.5,0.007692,0.020000,0.706667,0.861364,0.000000


In [400]:
raw_pw = (
    0.10 * df["stat_PW"]                    # tournament history
  + 0.25 * df["anytime_scorer_prob"]      # gets into dangerous areas
  + 0.10 * df["chance_created_per90"]     # attacking involvement
  + 0.15 * df["stat_CC"]                  # recent form in creation
  + 0.15 * df["team_score_2_prob"]        # team expected to attack heavily
  + 0.10 * df["price_rank_team_position"] # quality proxy
)

# Penalties are rare → heavily compress probabilities + minutes as external gate
df["prob_pen_won"] = 0.15 * sigmoid((raw_pw - 0.65) * 5) * df["prob_plays_60min"]

In [401]:
cols = [
    "name",
    "team",
    "position",
    "prob_pen_won",
    "stat_PW",
    "anytime_scorer_prob",
    "assist_expectation",
    "chance_created_per90",
    "team_score_2_prob",
    "price_rank_team_position",
    "prob_plays_60min",
]

df[cols].sort_values("prob_pen_won", ascending=False).head(30)

,name,team,position,prob_pen_won,stat_PW,anytime_scorer_prob,assist_expectation,chance_created_per90,team_score_2_prob,price_rank_team_position,prob_plays_60min
106,Kylian Mbappé,France,FWD,0.049460,0.0,1.000000,0.030294,0.028490,1.000000,1.000000,0.922048
119,Michael Olise,France,MID,0.044351,0.0,0.541660,0.172615,0.162338,1.000000,0.900000,0.803942
107,Ousmane Dembélé,France,MID,0.032764,0.0,0.708610,0.036920,0.034722,1.000000,1.000000,0.781460
27,Vinícius Júnior,Brazil,MID,0.031287,0.0,0.718813,0.027980,0.028490,0.704250,1.000000,0.873305
220,Mikel Oyarzabal,Spain,FWD,0.030898,0.0,0.736435,0.031814,0.033223,0.643539,1.000000,0.876473
144,Ismael Saibari,Morocco,MID,0.028857,0.0,0.562220,0.051223,0.055096,0.563053,1.000000,0.896428
18,Youri Tielemans,Belgium,MID,0.025339,1.0,0.322152,0.023914,0.025974,0.541891,0.700000,0.873335
157,Erling Haaland,Norway,FWD,0.024854,0.0,0.725769,0.029790,0.037037,0.338248,1.000000,0.851601
28,Matheus Cunha,Brazil,FWD,0.023028,0.0,0.633792,0.000000,0.000000,0.704250,1.000000,0.794758
10,Romelu Lukaku,Belgium,FWD,0.022964,0.0,0.718813,0.000000,0.000000,0.541891,1.000000,0.802544


In [402]:
raw_cs = (
    0.8 * df["team_cs_prob"]            # strongest signal
  + 0.10 * df["stat_CS"]               # tournament history
  + 0.10 * (1 - df["stat_GC"])         # fewer goals conceded is better
)

base_cs = sigmoid((raw_cs - 0.5) * 6)

df["prob_clean_sheet"] = base_cs * df["prob_plays_60min"]

df.loc[df["position"] == "FWD", "prob_clean_sheet"] = 0.0

In [403]:
cols = [
    "name",
    "team",
    "position",
    "prob_clean_sheet",
    "team_cs_prob",
    "prob_plays_60min",
    "stat_CS",
    "stat_GC",
    "minutes_rank_team_position",
    "tackles_per90",
]

df[cols].sort_values("prob_clean_sheet", ascending=False).head(30)

,name,team,position,prob_clean_sheet,team_cs_prob,prob_plays_60min,stat_CS,stat_GC,minutes_rank_team_position,tackles_per90
110,Mike Maignan,France,GK,0.853200,1.000000,0.921262,0.50,0.285714,1.000000,0.000000
101,Dayot Upamecano,France,DEF,0.783523,1.000000,0.846026,0.50,0.285714,1.000000,0.020231
261,Jules Koundé,France,DEF,0.749103,1.000000,0.808861,0.50,0.285714,0.877778,0.032836
119,Michael Olise,France,MID,0.744547,1.000000,0.803942,0.50,0.285714,1.000000,0.012987
107,Ousmane Dembélé,France,MID,0.735131,1.000000,0.781460,0.75,0.142857,0.900000,0.010417
69,Camilo Vargas,Colombia,GK,0.661550,0.846259,0.762867,0.50,0.142857,1.000000,0.000000
260,William Saliba,France,DEF,0.642594,1.000000,0.689645,0.50,0.142857,0.755556,0.007407
224,Unai Simón,Spain,GK,0.582435,0.493063,0.912904,1.00,0.000000,1.000000,0.000000
340,Yassine Bounou,Morocco,GK,0.575933,0.631567,0.947350,0.25,0.571429,1.000000,0.000000
65,Luis Díaz,Colombia,MID,0.574544,0.846259,0.662536,0.50,0.142857,1.000000,0.003717


In [404]:
position_rc_modifier = {
    "GK": 0.04,
    "FWD": 0.08,
    "DEF": 0.12,
    "MID": 0.14,
}

df["position_rc_modifier"] = df["position"].map(position_rc_modifier)

raw_rc = (
    0.30 * df["position_rc_modifier"]   # strongest prior
  + 0.25 * df["rc_per90"]              # direct history
  + 0.15 * df["tackles_per90"]         # physical involvement
  + 0.15 * df["prob_yellow_card"]      # disciplinary tendency
  + 0.10 * df["opp_over_05_prob"]      # more defending -> more challenges
)

# Red cards are extremely rare → strong compression + external gate
df["prob_red_card"] = 0.05 * sigmoid((raw_rc - 0.5) * 5) * df["prob_plays_60min"]

In [405]:
cols = [
    "name",
    "team",
    "position",
    "prob_red_card",
    "position_rc_modifier",
    "rc_per90",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_red_card", ascending=False).head(30)

,name,team,position,prob_red_card,position_rc_modifier,rc_per90,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
177,Miguel Almirón,Paraguay,MID,0.009245,0.14,0.728972,0.032710,0.099216,1.000000,0.604311
123,César Montes,Mexico,DEF,0.007806,0.12,0.577778,0.007407,0.064460,0.608231,0.565440
170,Gustavo Gómez,Paraguay,DEF,0.006923,0.12,0.000000,0.010256,0.098385,1.000000,0.926121
168,Matías Galarza,Paraguay,MID,0.006394,0.14,0.000000,0.046980,0.130667,1.000000,0.769872
284,Andrés Cubas,Paraguay,MID,0.006348,0.14,0.000000,0.035897,0.104211,1.000000,0.611035
186,Orlando Gill,Paraguay,GK,0.006137,0.04,0.000000,0.000000,0.090798,1.000000,0.902911
188,Julio Enciso,Paraguay,MID,0.006034,0.14,0.000000,0.012270,0.086163,1.000000,0.854177
191,Nuno Mendes,Portugal,DEF,0.005891,0.12,0.000000,0.014663,0.080690,0.803747,0.892531
18,Youri Tielemans,Belgium,MID,0.005829,0.14,0.000000,0.010390,0.077812,0.706667,0.828441
209,Bruno Fernandes,Portugal,MID,0.005803,0.14,0.000000,0.024024,0.079323,0.803747,0.883563


In [406]:
position_og_modifier = {
    "GK": 0.01,
    "FWD": 0.02,
    "MID": 0.04,
    "DEF": 0.08,
}

df["position_og_modifier"] = df["position"].map(position_og_modifier)

raw_og = (
    0.4 * df["position_og_modifier"]
  + 0.25 * df["opp_over_05_prob"]
  + 0.10 * df["tackles_per90"]
  + 0.10 * (1 - df["stat_GC"])
)

df["prob_own_goal"] = 0.03 * sigmoid((raw_og - 0.5) * 5) * df["prob_plays_60min"]

In [407]:
cols = [
    "name",
    "team",
    "position",
    "prob_own_goal",
    "position_og_modifier",
    "own_goal_rate",
    "opp_over_05_prob",
    "tackles_per90",
    "stat_GC",
    "prob_plays_any",
]

df[cols].sort_values("prob_own_goal", ascending=False).head(30)

,name,team,position,prob_own_goal,position_og_modifier,own_goal_rate,opp_over_05_prob,tackles_per90,stat_GC,prob_plays_any
168,Matías Galarza,Paraguay,MID,0.007851,0.04,0.0,1.000000,0.046980,0.142857,0.769872
170,Gustavo Gómez,Paraguay,DEF,0.007793,0.08,0.0,1.000000,0.010256,0.714286,0.926121
191,Nuno Mendes,Portugal,DEF,0.007126,0.08,0.0,0.803747,0.014663,0.285714,0.892531
186,Orlando Gill,Paraguay,GK,0.006964,0.01,0.0,1.000000,0.000000,0.714286,0.902911
278,Diogo Costa,Portugal,GK,0.006809,0.01,0.0,0.803747,0.000000,0.285714,0.937479
175,Juan José Cáceres,Paraguay,DEF,0.006756,0.08,0.0,1.000000,0.051576,0.571429,0.610705
282,Cristiano Ronaldo,Portugal,FWD,0.006706,0.02,0.0,0.803747,0.005698,0.285714,0.919309
284,Andrés Cubas,Paraguay,MID,0.006504,0.04,0.0,1.000000,0.035897,0.714286,0.611035
279,Rúben Dias,Portugal,DEF,0.006466,0.08,0.0,0.803747,0.018519,0.142857,0.737912
209,Bruno Fernandes,Portugal,MID,0.006439,0.04,0.0,0.803747,0.024024,0.285714,0.883563


In [408]:
position_pc_modifier = {
    "GK": 0.02,
    "FWD": 0.03,
    "MID": 0.06,
    "DEF": 0.10,
}

df["position_pc_modifier"] = df["position"].map(position_pc_modifier)

raw_pc = (
    0.30 * df["position_pc_modifier"]
  + 0.25 * df["penalty_conceded_rate"]
  + 0.05 * df["tackles_per90"]
  + 0.15 * df["prob_yellow_card"]
  + 0.15 * df["opp_over_05_prob"]
)

df["prob_penalty_committed"] = 0.08 * sigmoid((raw_pc - 0.5) * 5) * df["prob_plays_60min"]

In [409]:
cols = [
    "name",
    "team",
    "position",
    "prob_penalty_committed",
    "position_pc_modifier",
    "penalty_conceded_rate",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_penalty_committed", ascending=False).head(30)

,name,team,position,prob_penalty_committed,position_pc_modifier,penalty_conceded_rate,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
170,Gustavo Gómez,Paraguay,DEF,0.013256,0.10,0.0,0.010256,0.098385,1.000000,0.926121
186,Orlando Gill,Paraguay,GK,0.011846,0.02,0.0,0.000000,0.090798,1.000000,0.902911
168,Matías Galarza,Paraguay,MID,0.011178,0.06,0.0,0.046980,0.130667,1.000000,0.769872
284,Andrés Cubas,Paraguay,MID,0.011154,0.06,0.0,0.035897,0.104211,1.000000,0.611035
171,Júnior Alonso,Paraguay,DEF,0.010849,0.10,0.0,0.026667,0.098627,1.000000,0.621671
175,Juan José Cáceres,Paraguay,DEF,0.010849,0.10,0.0,0.051576,0.097152,1.000000,0.610705
191,Nuno Mendes,Portugal,DEF,0.010839,0.10,0.0,0.014663,0.080690,0.803747,0.892531
188,Julio Enciso,Paraguay,MID,0.010713,0.06,0.0,0.012270,0.086163,1.000000,0.854177
278,Diogo Costa,Portugal,GK,0.010387,0.02,0.0,0.000000,0.079504,0.803747,0.937479
303,Alistair Johnston,Canada,DEF,0.010319,0.10,0.0,0.002778,0.089558,0.760766,0.846049


In [410]:
df["lambda_opp"] = -np.log(df["team_cs_prob"].clip(lower=0.01))

df["prob_gc_0"] = np.exp(-df["lambda_opp"])
df["prob_gc_1"] = df["lambda_opp"] * np.exp(-df["lambda_opp"])
df["prob_gc_2"] = (df["lambda_opp"] ** 2 / 2) * np.exp(-df["lambda_opp"])
df["prob_gc_3"] = (df["lambda_opp"] ** 3 / 6) * np.exp(-df["lambda_opp"])
df["prob_gc_4plus"] = 1 - (
    df["prob_gc_0"]
    + df["prob_gc_1"]
    + df["prob_gc_2"]
    + df["prob_gc_3"]
)

# RAW expected goals conceded (true intensity)
df["expected_goals_conceded"] = df["lambda_opp"]

# Normalize ONLY as auxiliary feature (separate column)
scaler = MinMaxScaler()
df["expected_goals_conceded_norm"] = scaler.fit_transform(
    df[["expected_goals_conceded"]]
)

# Player exposure to goals conceded (correct)
df["expected_player_goals_conceded"] = (
    df["lambda_opp"] * df["prob_plays_60min"]
)

# GC penalty MUST use raw λ (correct Poisson expectation)
df["expected_gc_penalty"] = (
    -(df["lambda_opp"] - 1 + np.exp(-df["lambda_opp"]))
    * df["prob_plays_60min"]
)

df.loc[df["position"].isin(["MID", "FWD"]), [
    "expected_player_goals_conceded",
    "expected_gc_penalty"
]] = 0.0

In [411]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "team_cs_prob",
    "lambda_opp",
    "prob_gc_0",
    "prob_gc_1",
    "prob_gc_2",
    "prob_gc_3",
    "prob_gc_4plus",
    "expected_player_goals_conceded",
    "expected_gc_penalty",
]

df[df["position"].isin(["GK", "DEF"])][cols] \
    .sort_values("expected_gc_penalty") \
    .head(30)

,name,team,position,prob_plays_60min,team_cs_prob,lambda_opp,prob_gc_0,prob_gc_1,prob_gc_2,prob_gc_3,prob_gc_4plus,expected_player_goals_conceded,expected_gc_penalty
170,Gustavo Gómez,Paraguay,DEF,0.926121,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,4.264943,-3.348084
186,Orlando Gill,Paraguay,GK,0.920561,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,4.239342,-3.327986
171,Júnior Alonso,Paraguay,DEF,0.755296,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,3.478267,-2.730524
175,Juan José Cáceres,Paraguay,DEF,0.752093,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,3.463517,-2.718945
172,Omar Alderete,Paraguay,DEF,0.461743,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,2.126403,-1.669278
176,José Canale,Paraguay,DEF,0.337258,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,1.553130,-1.219244
174,Gustavo Velázquez,Paraguay,DEF,0.310458,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,1.429713,-1.122359
184,Roberto Fernández,Paraguay,GK,0.266002,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,1.224987,-0.961644
161,Ørjan Nyland,Norway,GK,0.863358,0.147601,1.913245,0.147601,0.282396,0.270147,0.172286,0.127571,1.651815,-0.915889
120,Benjamin Asare,Ghana,GK,0.672692,0.115078,2.162141,0.115078,0.248816,0.268988,0.193863,0.173255,1.454454,-0.859175


In [412]:
raw_save = (
    0.25 * df["expected_goals_conceded"]  # opportunity
  + 0.20 * df["saves_per90"]             # ability
  + 0.05 * df["recent_saves_per90"]      # recent form
  + 0.20 * df["price"]                   # overall GK quality
)

# Expected saves (λ)
df["expected_saves"] = (
    sigmoid((raw_save - 0.5) * 6) * 4
) * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "expected_saves"] = 0.0



In [413]:
cols = [
    "name",
    "team",
    "expected_saves",
    "expected_goals_conceded",
    "saves_per90",
    "recent_saves_per90",
    "price",
    "prob_plays_any",
]

df[df["position"] == "GK"][cols] \
    .sort_values("expected_saves", ascending=False) \
    .head(30)

,name,team,expected_saves,expected_goals_conceded,saves_per90,recent_saves_per90,price,prob_plays_any
186,Orlando Gill,Paraguay,3.662987,4.605170,0.876923,0.960000,0.000000,0.902911
278,Diogo Costa,Portugal,2.553604,1.649341,0.700000,0.866667,0.200000,0.937479
161,Ørjan Nyland,Norway,2.268837,1.913245,0.400000,0.600000,0.100000,0.896671
120,Benjamin Asare,Ghana,2.073951,2.162141,0.640000,0.500000,0.042857,0.813408
272,Thibaut Courtois,Belgium,1.630302,1.236409,0.415385,0.420000,0.200000,0.947350
247,Matt Freese,USA,1.465280,1.349894,0.333333,0.500000,0.100000,0.896671
49,Maxime Crépeau,Canada,1.450981,1.415135,0.250000,0.200000,0.071429,0.937479
162,Egil Selvik,Norway,1.168544,1.913245,1.000000,1.000000,0.042857,0.449405
35,Alisson Becker,Brazil,1.147800,0.753374,0.550000,0.600000,0.214286,0.923274
184,Roberto Fernández,Paraguay,1.044749,4.605170,0.000000,0.000000,0.071429,0.463813


In [414]:
raw_pen_save = (
    0.40 * df["price"]                    
  + 0.25 * df["expected_goals_conceded"]
  + 0.15 * df["opp_over_05_prob"]
  + 0.05 * df["stat_PS"]
)

# Strong compression because penalty saves are exceptionally rare
df["prob_penalty_save"] = sigmoid((raw_pen_save - 0.5) * 3) / 20 * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "prob_penalty_save"] = 0.0

In [415]:
cols = [
    "name",
    "team",
    "prob_penalty_save",
    "price",
    "expected_goals_conceded",
    "opp_over_05_prob",
    "prob_plays_60min",
    "stat_PS",
]

df[df["position"] == "GK"][cols] \
    .sort_values("prob_penalty_save", ascending=False) \
    .head(30)

,name,team,prob_penalty_save,price,expected_goals_conceded,opp_over_05_prob,prob_plays_60min,stat_PS
186,Orlando Gill,Paraguay,0.042213,0.000000,4.605170,1.000000,0.920561,0.0
278,Diogo Costa,Portugal,0.026895,0.200000,1.649341,0.803747,0.921262,0.0
161,Ørjan Nyland,Norway,0.026226,0.100000,1.913245,0.848970,0.863358,0.0
272,Thibaut Courtois,Belgium,0.023510,0.200000,1.236409,0.706667,0.947350,0.0
49,Maxime Crépeau,Canada,0.022910,0.071429,1.415135,0.760766,0.921262,0.0
120,Benjamin Asare,Ghana,0.021479,0.042857,2.162141,0.880453,0.672692,0.0
247,Matt Freese,USA,0.021244,0.100000,1.349894,0.746908,0.863358,0.0
35,Alisson Becker,Brazil,0.017807,0.214286,0.753374,0.531328,0.908430,0.0
224,Unai Simón,Spain,0.017419,0.214286,0.707117,0.510749,0.912904,0.0
135,Raúl Rangel,Mexico,0.016982,0.057143,0.939180,0.608231,0.874073,0.0


In [416]:
raw_tackles = (
    0.40 * df["tackles_per90"]
  + 0.20 * df["recent_tackles_per90"]
  + 0.05 * df["expected_goals_conceded"]
  + 0.10 * df["match_over_25_prob"]
  + 0.15 * df["price"]
)

lam_tackles = sigmoid((raw_tackles - 0.5) * 6) * 3
df["expected_tackle_points"] = (lam_tackles / 3) * df["prob_plays_60min"]
df.loc[df["position"] != "MID", "expected_tackle_points"] = 0.0

In [417]:
cols = [
    "name",
    "team",
    "expected_tackle_points",
    "tackles_per90",
    "recent_tackles_per90",
    "expected_goals_conceded",
    "match_over_25_prob",
    "price",
    "prob_plays_60min",
]

df[df["position"] == "MID"][cols] \
    .sort_values("expected_tackle_points", ascending=False) \
    .head(30)

,name,team,expected_tackle_points,tackles_per90,recent_tackles_per90,expected_goals_conceded,match_over_25_prob,price,prob_plays_60min
168,Matías Galarza,Paraguay,0.320275,0.046980,0.281879,4.605170,1.000000,0.185714,0.798212
284,Andrés Cubas,Paraguay,0.294653,0.035897,0.180000,4.605170,1.000000,0.171429,0.811533
188,Julio Enciso,Paraguay,0.293598,0.012270,0.050847,4.605170,1.000000,0.442857,0.792108
177,Miguel Almirón,Paraguay,0.242876,0.032710,0.222222,4.605170,1.000000,0.357143,0.588171
189,Diego Gómez,Paraguay,0.209887,0.033898,0.230769,4.605170,1.000000,0.471429,0.475609
187,Damián Bobadilla,Paraguay,0.186194,0.036842,0.289655,4.605170,1.000000,0.285714,0.443970
209,Bruno Fernandes,Portugal,0.179227,0.024024,0.123457,1.649341,0.622824,0.714286,0.825673
293,Martin Ødegaard,Norway,0.176394,0.022989,0.133333,1.913245,0.831077,0.600000,0.745991
27,Vinícius Júnior,Brazil,0.172980,0.005698,0.022989,0.753374,0.831077,0.928571,0.873305
234,Weston McKennie,USA,0.155058,0.022599,0.181818,1.349894,0.831077,0.371429,0.842843


In [418]:
raw_cc = (
    0.20 * df["cc_per90"]
  + 0.10 * df["recent_cc_per90"]
  + 0.10 * df["stat_CC"]
  + 0.25 * df["match_over_25_prob"]
  + 0.10 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)
lam_cc = sigmoid((raw_cc - 0.5) * 6) * 3
df["expected_cc_points"] = (lam_cc / 2) * df["prob_plays_60min"]
df.loc[df["position"] != "MID", "expected_cc_points"] = 0.0

In [419]:
cols = [
    "name",
    "team",
    "position",
    "expected_cc_points",
    "cc_per90",
    "recent_cc_per90",
    "stat_CC",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_cc_points", ascending=False).head(30)

,name,team,position,expected_cc_points,cc_per90,recent_cc_per90,stat_CC,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
119,Michael Olise,France,MID,0.675390,0.162338,0.178899,1.0,1.000000,0.541660,0.857143,0.803942
107,Ousmane Dembélé,France,MID,0.493533,0.034722,0.062500,0.2,1.000000,0.708610,0.928571,0.781460
27,Vinícius Júnior,Brazil,MID,0.469921,0.028490,0.049808,0.2,0.831077,0.718813,0.928571,0.873305
276,Leandro Trossard,Belgium,MID,0.382458,0.055402,0.095941,0.4,0.831077,0.431288,0.442857,0.860441
188,Julio Enciso,Paraguay,MID,0.378749,0.061350,0.055085,0.4,1.000000,0.213634,0.442857,0.792108
252,Bruno Guimarães,Brazil,MID,0.376119,0.088235,0.150000,0.6,0.831077,0.227392,0.471429,0.796764
293,Martin Ødegaard,Norway,MID,0.329291,0.076628,0.072222,0.4,0.831077,0.238986,0.600000,0.745991
262,Bradley Barcola,France,MID,0.329213,0.048077,0.065657,0.2,1.000000,0.579688,0.642857,0.599648
209,Bruno Fernandes,Portugal,MID,0.313586,0.060060,0.106996,0.4,0.622824,0.298810,0.714286,0.825673
18,Youri Tielemans,Belgium,MID,0.306492,0.025974,0.000000,0.2,0.831077,0.322152,0.371429,0.873335


In [420]:
raw_sot = (
    0.35 * df["sot_per90"]
  + 0.20 * df["goal_rate"]
  + 0.15 * df["stat_GS"]
  + 0.20 * df["match_over_25_prob"]
  + 0.10 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)
lam_sot = sigmoid((raw_sot - 0.5) * 6) * 3  # expected SoT volume per game
df["expected_sot_points"] = (lam_sot / 2) * df["prob_plays_60min"]
df.loc[~df["position"].isin(["FWD"]), "expected_sot_points"] = 0.0

In [421]:
cols = [
    "name",
    "team",
    "position",
    "expected_sot_points",
    "sot_per90",
    "goal_rate",
    "stat_GS",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_sot_points", ascending=False).head(30)

,name,team,position,expected_sot_points,sot_per90,goal_rate,stat_GS,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
106,Kylian Mbappé,France,FWD,1.179724,0.518519,0.307692,1.000000,1.000000,1.000000,1.000000,0.922048
157,Erling Haaland,Norway,FWD,0.973166,0.466667,0.333333,0.833333,0.831077,0.725769,1.000000,0.851601
220,Mikel Oyarzabal,Spain,FWD,0.739737,0.372093,0.239203,0.666667,0.622824,0.736435,0.657143,0.876473
282,Cristiano Ronaldo,Portugal,FWD,0.627510,0.279202,0.153846,0.500000,0.622824,0.584171,0.928571,0.891920
28,Matheus Cunha,Brazil,FWD,0.612324,0.297872,0.229787,0.500000,0.831077,0.633792,0.542857,0.794758
85,Harry Kane,England,FWD,0.609149,0.355932,0.254237,0.833333,0.000000,0.633792,1.000000,0.910706
10,Romelu Lukaku,Belgium,FWD,0.494468,0.158192,0.203390,0.333333,0.831077,0.718813,0.557143,0.802544
44,Jonathan David,Canada,FWD,0.363252,0.296073,0.163142,0.500000,0.191489,0.376101,0.500000,0.885500
128,Julián Quiñones,Mexico,FWD,0.221873,0.210210,0.162162,0.500000,0.000000,0.431288,0.300000,0.794582
197,Gonçalo Ramos,Portugal,FWD,0.213910,0.411765,0.529412,0.166667,0.622824,0.488329,0.571429,0.280551


In [422]:
df["expected_qualification_points"] = df["team_qualify_prob"] * 2

In [423]:
cols = [
    "name",
    "team",
    "position",
    "expected_qualification_points",
    "team_qualify_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_qualification_points", ascending=False).head(30)

,name,team,position,expected_qualification_points,team_qualify_prob,price,prob_plays_60min
100,Maxence Lacroix,France,DEF,2.000000,1.000000,0.142857,0.140576
110,Mike Maignan,France,GK,2.000000,1.000000,0.214286,0.921262
111,Brice Samba,France,GK,2.000000,1.000000,0.142857,0.206689
112,N'Golo Kanté,France,MID,2.000000,1.000000,0.228571,0.059524
113,Adrien Rabiot,France,MID,2.000000,1.000000,0.414286,0.586244
114,Warren Zaïre-Emery,France,MID,2.000000,1.000000,0.371429,0.060370
263,Jean-Philippe Mateta,France,FWD,2.000000,1.000000,0.428571,0.212867
99,Theo Hernández,France,DEF,2.000000,1.000000,0.214286,0.363809
116,Maghnes Akliouche,France,MID,2.000000,1.000000,0.414286,0.082256
115,Manu Koné,France,MID,2.000000,1.000000,0.371429,0.226586


In [424]:
df["ep_appearance"] = df["prob_plays_any"] + df["prob_plays_60min"]

GOAL_PTS = {"GK": 9, "DEF": 7, "MID": 6, "FWD": 5}
df["ep_goal"] = df["prob_scores"] * df["position"].map(GOAL_PTS)

df["ep_assist"] = df["expected_assists"] * 3

df["ep_yellow_card"]  = df["prob_yellow_card"]        * (-1)
df["ep_red_card"]     = df["prob_red_card"]           * (-2)
df["ep_own_goal"]     = df["prob_own_goal"]           * (-2)
df["ep_pen_won"]      = df["prob_pen_won"]            *   2
df["ep_pen_conceded"] = df["prob_penalty_committed"]  * (-1)


CS_PTS = {"GK": 5, "DEF": 5, "MID": 1, "FWD": 0}
df["ep_clean_sheet"] = df["prob_clean_sheet"] * df["position"].map(CS_PTS)



df["ep_gc_penalty"] = df["expected_gc_penalty"]       # already zeroed for MID/FWD

df["ep_penalty_save"] = 0.0
df["ep_saves"]        = 0.0
gk = df["position"] == "GK"
df.loc[gk, "ep_penalty_save"] = df.loc[gk, "prob_penalty_save"] * 3
df.loc[gk, "ep_saves"]        = df.loc[gk, "expected_saves"] / 3

df["ep_tackles"] = 0.0
df["ep_cc"]      = 0.0
mid = df["position"] == "MID"
df.loc[mid, "ep_tackles"] = df.loc[mid, "expected_tackle_points"]   # already /3
df.loc[mid, "ep_cc"]      = df.loc[mid, "expected_cc_points"]       # already /2

df["ep_sot"] = 0.0
fwd = df["position"] == "FWD"
df.loc[fwd, "ep_sot"] = df.loc[fwd, "expected_sot_points"]         # already /2


df["ep_qualification"] = df["expected_qualification_points"] * df["prob_plays_any"]

EP_COLS = [
    "ep_appearance",
    "ep_goal",
    "ep_assist",
    "ep_yellow_card",
    "ep_red_card",
    "ep_own_goal",
    "ep_pen_won",
    "ep_pen_conceded",
    "ep_clean_sheet",
    "ep_gc_penalty",
    "ep_penalty_save",
    "ep_saves",
    "ep_tackles",
    "ep_cc",
    "ep_sot",
    "ep_qualification",
]
df["expected_points"] = df[EP_COLS].sum(axis=1)

In [425]:
EP_COLS = [c for c in df.columns if c.startswith("ep_") and c != "ep_scouting"]
df["expected_points"] = df[EP_COLS].sum(axis=1)
df.to_parquet("fantasy_optimizer.parquet", index=False)

In [426]:
df.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {df.shape[0]} rows x {df.shape[1]} cols")


Saved fantasy_enriched.csv 362 rows x 235 cols


In [ ]:
# ── Scouting bonus expected value ─────────────────────────────────────────────
# Rule: +2 if player scores >4 BASE points AND is in <5% of teams.
# Ownership is known pre-round. The >4pt threshold requires a probability
# estimate: model points as Poisson(mu=expected_points) — standard in soccer
# fantasy literature. E[scouting] = I(ownership<5%) * P(X>4|mu) * 2.

def add_scouting_ep(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["scouting_eligible"] = (df["percent_selected"] < 5.0).astype(float)
    df["prob_exceeds_4pts"] = df["expected_points"].apply(
        lambda mu: float(1 - poisson.cdf(4, max(mu, 1e-9)))
    )
    df["ep_scouting"] = df["scouting_eligible"] * df["prob_exceeds_4pts"] * 2
    return df


# ── ILP Optimizer ─────────────────────────────────────────────────────────────

def optimize_squad(
    df: pd.DataFrame,
    budget: float = 100.0,        # $100M group stage | $105M knockout
    max_per_country: int = 3,     # 3 group/R32 | 4 R16 | 5 QF | 6 SF | 8 F
    vc_dnp_prob: float = 0.10,    # P(captain DNP): approximation for VC bonus
    bench_weight: float = 0.05,   # discount factor for bench auto-sub value
    solver=None,
) -> dict | None:
    """
    Integer linear programme solving the FIFA WC Fantasy 2026 squad selection.

    Decision variables
    ─────────────────
    x[i] ∈ {0,1}  player i in 15-man squad
    s[i] ∈ {0,1}  player i in starting XI          (implies x[i]=1)
    b[i] ∈ {0,1}  player i on bench                (x[i] = s[i] + b[i])
    c[i] ∈ {0,1}  player i is captain              (must be in XI)
    v[i] ∈ {0,1}  player i is vice-captain         (must be in XI, ≠ captain)

    Objective (all in expected-points space)
    ──────────────────────────────────────
    max  Σ ep[i]·s[i]                              # XI base
       + Σ ep[i]·c[i]                              # captain doubling extra
       + Σ ep[i]·vc_dnp_prob·v[i]                 # VC expected extra (approx)
       + Σ ep_scouting[i]·s[i]                    # scouting bonus
       + Σ ep[i]·bench_weight·b[i]                # bench auto-sub upside
    """
    if "ep_scouting" not in df.columns:
        df = add_scouting_ep(df)

    idx = df.index.tolist()

    # ── Decision variables ────────────────────────────────────────────────────
    x = LpVariable.dicts("squad", idx, cat="Binary")
    s = LpVariable.dicts("xi",    idx, cat="Binary")
    b = LpVariable.dicts("bench", idx, cat="Binary")
    c = LpVariable.dicts("cap",   idx, cat="Binary")
    v = LpVariable.dicts("vc",    idx, cat="Binary")

    model = LpProblem("FIFA_WC_Fantasy_2026", LpMaximize)

    # ── Objective ─────────────────────────────────────────────────────────────
    model += lpSum(
        df.loc[i, "expected_points"]                          * s[i]
        + df.loc[i, "expected_points"]                        * c[i]
        + df.loc[i, "expected_points"] * vc_dnp_prob          * v[i]
        + df.loc[i, "ep_scouting"]                            * s[i]
        + df.loc[i, "expected_points"] * bench_weight          * b[i]
        for i in idx
    )

    # ── Squad size ────────────────────────────────────────────────────────────
    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]   # partition: every squad player is XI or bench

    # ── Budget ────────────────────────────────────────────────────────────────
    model += lpSum(df.loc[i, "price"] * x[i] for i in idx) <= budget

    # ── Squad position composition (15-man) ───────────────────────────────────
    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    # ── Country limit ─────────────────────────────────────────────────────────
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        if c_idx:
            model += lpSum(x[i] for i in c_idx) <= max_per_country

    # ── Starting XI: goalkeeper ───────────────────────────────────────────────
    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1    # exactly 1 GK starts
    model += lpSum(b[i] for i in gk_idx) == 1    # exactly 1 GK on bench

    # ── Starting XI: valid outfield formation ─────────────────────────────────
    # The 7 permitted formations are: 4-4-2, 4-3-3, 4-5-1, 3-4-3,
    # 3-5-2, 5-4-1, 5-3-2. All satisfy:
    #   DEF ∈ [3,5], MID ∈ [3,5], FWD ∈ [1,3], DEF+MID+FWD = 10 (implicit).
    # These bounds are tight — no invalid combination passes both constraints.
    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    # ── Captain ───────────────────────────────────────────────────────────────
    model += lpSum(c[i] for i in idx) == 1
    for i in idx:
        model += c[i] <= s[i]       # captain must be in XI

    # ── Vice-captain ──────────────────────────────────────────────────────────
    model += lpSum(v[i] for i in idx) == 1
    for i in idx:
        model += v[i] <= s[i]       # VC must be in XI
        model += c[i] + v[i] <= 1  # captain ≠ vice-captain

    # ── Solve ─────────────────────────────────────────────────────────────────
    solver = solver or PULP_CBC_CMD(msg=0)
    model.solve(solver)

    if LpStatus[model.status] != "Optimal":
        print(f"[!] Solver status: {LpStatus[model.status]}")
        return None

    # ── Extract solution ──────────────────────────────────────────────────────
    in_squad = [i for i in idx if value(x[i]) > 0.5]
    in_xi    = [i for i in idx if value(s[i]) > 0.5]
    on_bench = [i for i in idx if value(b[i]) > 0.5]
    captain  = next(i for i in idx if value(c[i]) > 0.5)
    vc       = next(i for i in idx if value(v[i]) > 0.5)

    return {
        "squad":    df.loc[in_squad].copy(),
        "xi":       df.loc[in_xi].copy(),
        "bench":    df.loc[on_bench].copy(),
        "captain":  captain,
        "vc":       vc,
        "total_ep": value(model.objective),
        "cost":     df.loc[in_squad, "price"].sum(),
        "status":   LpStatus[model.status],
    }


# ── Display ───────────────────────────────────────────────────────────────────

POS_ORDER = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}

def print_solution(result: dict, show_bench: bool = True) -> None:
    if result is None:
        print("No solution found.")
        return

    xi    = result["xi"].copy()
    bench = result["bench"].copy()
    cap   = result["captain"]
    vc    = result["vc"]

    xi["_ord"]    = xi["position"].map(POS_ORDER)
    bench["_ord"] = bench["position"].map(POS_ORDER)
    xi    = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
    bench = bench.sort_values(["_ord", "expected_points"], ascending=[True, False])

    def row_str(i, row):
        tag   = " [C]" if i == cap else " [V]" if i == vc else ""
        scout = " ★SCOUT" if row.get("scouting_eligible", 0) > 0.5 else ""
        return (f"  {row['position']:3}  {row['name']:<28} {row['team']:<22}"
                f"${row['price']:.1f}  EP:{row['expected_points']:.2f}{tag}{scout}")

    w = 72
    print(f"\n{'═'*w}")
    print(f"  EXPECTED POINTS: {result['total_ep']:.2f}   COST: ${result['cost']:.1f}M")
    print(f"{'═'*w}")

    print(f"\n  STARTING XI")
    print(f"  {'─'*68}")
    for i, row in xi.iterrows():
        print(row_str(i, row))

    if show_bench:
        print(f"\n  BENCH  (auto-sub priority: 1 → 4)")
        print(f"  {'─'*68}")
        for rank, (i, row) in enumerate(bench.iterrows(), 1):
            print(f"  [{rank}]" + row_str(i, row)[4:])

    print(f"\n  Country breakdown:")
    counts = result["squad"]["team"].value_counts()
    for team, n in counts.items():
        print(f"    {team}: {n}")
    print(f"{'═'*w}\n")


# ── Usage ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    df = pd.read_parquet("fantasy_optimizer.parquet")
    df = df[df["status"] == "playing"].copy()

    # expected_points must exist before add_scouting_ep
    # If it's not in the parquet, compute it here:
    EP_COLS = [c for c in df.columns if c.startswith("ep_") and c != "ep_scouting"]
    assert EP_COLS, f"No ep_ columns found. Run the EP computation cells first. Columns: {df.columns.tolist()}"
    df["expected_points"] = df[EP_COLS].sum(axis=1)

    df = add_scouting_ep(df)
    result = optimize_squad(df, budget=105.0, max_per_country=4)
    print_solution(result)

    # Round of 16 (budget +5M, country limit +1)
    # result = optimize_squad(df, budget=105.0, max_per_country=4)



════════════════════════════════════════════════════════════════════════
  EXPECTED POINTS: 118.63   COST: $8.2M
════════════════════════════════════════════════════════════════════════

  STARTING XI
  ────────────────────────────────────────────────────────────────────
  GK   Camilo Vargas                Colombia              $0.1  EP:7.26 ★SCOUT
  DEF  Dayot Upamecano              France                $0.3  EP:8.74 ★SCOUT
  DEF  Achraf Hakimi                Morocco               $0.4  EP:8.14 ★SCOUT
  DEF  Marc Cucurella               Spain                 $0.2  EP:7.22 ★SCOUT
  DEF  Aymeric Laporte              Spain                 $0.3  EP:6.52 ★SCOUT
  MID  Michael Olise                France                $0.9  EP:8.80 [V] ★SCOUT
  MID  Ousmane Dembélé              France                $0.9  EP:8.51 ★SCOUT
  MID  Vinícius Júnior              Brazil                $0.9  EP:8.08 ★SCOUT
  MID  Ismael Saibari               Morocco               $0.5  EP:6.86 ★SCOUT
  FWD  Kylia